In [0]:
# Step 1: Create the catalog and schema if they don't exist
print("Setting up catalog and schema...")
spark.sql("CREATE CATALOG IF NOT EXISTS dev_finance")
spark.sql("CREATE SCHEMA IF NOT EXISTS dev_finance.staging")
print("✓ Catalog and schema ready")

# Step 2: Create the volume if it doesn't exist
print("\nCreating volume...")
spark.sql("""
    CREATE VOLUME IF NOT EXISTS dev_finance.staging.data_export
    COMMENT 'Volume for storing media customer reviews in multiple formats'
""")
print("✓ Volume created: dev_finance.staging.data_export")

# Step 3: Read data from source table
print("\nReading source data...")
df = spark.table("samples.bakehouse.media_customer_reviews")
record_count = df.count()
print(f"Total records: {record_count}")
df.printSchema()

# Define volume base path
volume_base_path = "/Volumes/dev_finance/staging/data_export"

# Step 4: Write to CSV format
csv_path = f"{volume_base_path}/csv"
print(f"\nWriting to CSV: {csv_path}")
df.write.mode("overwrite").option("header", "true").csv(csv_path)
print("✓ CSV files created successfully")

# Step 5: Write to JSON format
json_path = f"{volume_base_path}/json"
print(f"\nWriting to JSON: {json_path}")
df.write.mode("overwrite").json(json_path)
print("✓ JSON files created successfully")

# Step 6: Write to Parquet format
parquet_path = f"{volume_base_path}/parquet"
print(f"\nWriting to Parquet: {parquet_path}")
df.write.mode("overwrite").parquet(parquet_path)
print("✓ Parquet files created successfully")

print("\n" + "="*60)
print("✓ Data successfully written to all three formats!")
print("="*60)
print(f"Records exported: {record_count}")
print(f"CSV:     {csv_path}")
print(f"JSON:    {json_path}")
print(f"Parquet: {parquet_path}")

In [0]:
# Verify data was written correctly by reading samples from each format

volume_base_path = "/Volumes/dev_finance/staging/data_export"

# Read and display sample from CSV
print("Sample from CSV format:")
print("="*60)
csv_df = spark.read.option("header", "true").csv(f"{volume_base_path}/csv")
print(f"Record count: {csv_df.count()}")
display(csv_df.limit(5))

print("\nSample from JSON format:")
print("="*60)
json_df = spark.read.json(f"{volume_base_path}/json")
print(f"Record count: {json_df.count()}")
display(json_df.limit(5))

print("\nSample from Parquet format:")
print("="*60)
parquet_df = spark.read.parquet(f"{volume_base_path}/parquet")
print(f"Record count: {parquet_df.count()}")
display(parquet_df.limit(5))